## Preliminares

In [ ]:
# Importar modulos y funciones necesarias
import pandas as pd
import numpy as np
import os
from datetime import datetime
from statsmodels.formula.api import ols
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from funciones import *

In [2]:
# Abrir archivo clean_data
data_folder = "../data"
df = pd.read_parquet(f"{data_folder}/clean_data.parquet")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78535 entries, 0 to 78534
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   YearsSinceAdded                 78535 non-null  float64       
 1   Beta                            78535 non-null  float64       
 2   operatingMargins                78535 non-null  float64       
 3   profitMargins                   78535 non-null  float64       
 4   ReturnOnAssets                  78535 non-null  float64       
 5   PriceToBook_Transformed         78535 non-null  float64       
 6   returnOnEquity_Transformed      78535 non-null  float64       
 7   PE_Trailing_Transformed         78535 non-null  float64       
 8   EnterpriseToEbitda_Transformed  78535 non-null  float64       
 9   MarketCap_log                   78535 non-null  float64       
 10  EnterpriseValue_log             78535 non-null  float64       
 11  de

# Modelo de ensamblado de árboles RandomForest

In [ ]:
# Ratios de valuación
ratios = ['PriceToBook_Transformed', 'PE_Trailing_Transformed', 'EnterpriseToEbitda_Transformed']

# Se asegura el ordenamiento por fecha
df.sort_values(by='Fecha', inplace=True)

# Se define la variable objetivo y las variables predictoras
label = 'EnterpriseToEbitda_Transformed'
X = df.drop(columns=ratios + ['Ticker', 'Fecha']) # Se quitan los ratios de valuación restantes, Ticker y Fecha de las variables predictoras
y = df[label]

# identificar columnas numéricas y categóricas
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()

# preprocesador: escala numéricas y codifica categóricas
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

# Particionar en train y test (80% train, 20% test) sin mezclar (shuffle=False) para respetar la secuencia temporal
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False, random_state=42)

pipe = Pipeline([
    ('pre', preprocessor),
    ('model', RandomForestRegressor(
        random_state=42,
        n_estimators=300,
        max_depth=25,
        min_samples_leaf=1,
        max_features='sqrt',
        max_samples= None        
        ))
])

print("Entrenando el modelo...")
pipe.fit(X_train, y_train)
# comparar predicciones con el conjunto test

y_pred = pipe.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MSE: 0.004605935347993065
R2: 0.7806238506569727


In [4]:
# Re-entrenar el modelo con todos los datos
print("Re-entrenando el modelo con el dataset completo...")

X_completo = df.drop([label,'Ticker'], axis=1)
y_completo = df[label]

pipe_final = Pipeline([
    ('pre', preprocessor),
    ('model', RandomForestRegressor(
        random_state=42,
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=3,
        min_samples_split=5,
        max_features=0.5,
        max_samples= 0.7        
        ))
])

pipe_final.fit(X_completo, y_completo)
r2_final = pipe_final.score(X_completo, y_completo)
print(f"R² sobre dataset completo: {r2_final:.4f}")

Re-entrenando el modelo con el dataset completo...
R² sobre dataset completo: 0.8261


In [ ]:
# Test de validación cruzada
# Configurar el validador de series temporales
tscv = TimeSeriesSplit(n_splits=5) # n_splits=5 creará 5 particiones temporales secuenciales

# 3. Test de validación cruzada temporal
cross_val_scores = cross_val_score(
    estimator=pipe_final, 
    X=X_completo, 
    y=y_completo, 
    cv=tscv,         
    scoring='r2',
    n_jobs=-1        
)

print(f"R² promedio Time Series CV: {cross_val_scores.mean():.4f} ± {cross_val_scores.std():.4f}")

R² promedio Time Series CV: 0.6512 ± 0.0389


In [6]:
# Importancia de factores en el modelo
rf_model = pipe_final.named_steps['model']
importances = rf_model.feature_importances_

# Obtener los nombres de las características después del preprocesamiento
preprocessor = pipe_final.named_steps['pre']
feature_names = preprocessor.get_feature_names_out()

feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)
feature_importance_df.head(20)

,feature,importance
4,num__ReturnOnAssets,0.136673
0,num__YearsSinceAdded,0.132304
2,num__operatingMargins,0.127713
3,num__profitMargins,0.112507
6,num__MarketCap_log,0.111464
8,num__debtToEquity_log,0.093424
1,num__Beta,0.085273
5,num__returnOnEquity_Transformed,0.082291
9,num__currentRatio_log,0.076944
7,num__EnterpriseValue_log,0.041407


## Aplicacion del modelo

Se generan 2 clusters segun las predicciones:
* Overprediction: si los residuos son mayores o iguales a cero.
*  Underprediction: si los residuos son menores a cero.

In [ ]:
# Recuperar el ticker usando el indice de y_test
tickers_test = df.loc[y_test.index, 'Ticker']

# Consolidar resultados individuales en un DataFrame
resultados_df = pd.DataFrame({
    'Ticker': tickers_test,
    'Real': y_test,
    'Predicho': y_pred
})

# Calcular el residuo para cada predicción
resultados_df['Residuo'] = resultados_df['Predicho'] - resultados_df['Real']

# Agrupar por ticker
resultados_agrupados = resultados_df.groupby('Ticker')[['Real', 'Predicho', 'Residuo']].mean()

# Generar el Cluster sobre el promedio de los residuos
resultados_agrupados['Cluster'] = ['Overprediction' if r >= 0 else 'Underprediction' 
                                   for r in resultados_agrupados['Residuo']]

# Visualizar
fig = px.scatter(
    resultados_agrupados, 
    x='Real', 
    y='Predicho', 
    color='Cluster',
    hover_name=resultados_agrupados.index, 
    labels={'Real':'Valor Real Promedio', 'Predicho':'Predicción Promedio', 'Cluster':'Sesgo del Modelo'},
    title='Predicciones vs Reales (Agrupado por Ticker)'
)

# Línea de identidad perfecta
min_val = resultados_agrupados['Real'].min()
max_val = resultados_agrupados['Real'].max()
fig.add_shape(type='line', x0=min_val, y0=min_val, x1=max_val, y1=max_val,
              line=dict(color='black', dash='dash', width=2))
fig.show()

# Estadísticas por cluster a nivel Ticker
over_mask = resultados_agrupados['Cluster'] == 'Overprediction'
under_mask = resultados_agrupados['Cluster'] == 'Underprediction'

print("\nEstadísticas por cluster (a nivel de Ticker):")
print(f"Overprediction: {over_mask.sum()} tickers, residuo medio global: {resultados_agrupados.loc[over_mask, 'Residuo'].mean():.4f}")
print(f"Underprediction: {under_mask.sum()} tickers, residuo medio global: {resultados_agrupados.loc[under_mask, 'Residuo'].mean():.4f}")


Estadísticas por cluster (a nivel de Ticker):
Overprediction: 255 tickers, residuo medio global: 0.0264
Underprediction: 200 tickers, residuo medio global: -0.0364


In [12]:
# Ordenar resultados por residuos
resultados_agrupados = resultados_agrupados.sort_values(by='Residuo', ascending=False)
resultados_agrupados.head(20)

,Real,Predicho,Residuo,Cluster
Ticker,,,,
CVNA,-0.089362,0.098252,0.187614,Overprediction
SNDK,-0.484061,-0.301233,0.182827,Overprediction
XYZ,-0.078030,0.047269,0.125299,Overprediction
KHC,-0.191786,-0.087921,0.103865,Overprediction
KKR,-0.086597,0.017210,0.103807,Overprediction
WBD,-0.168673,-0.067891,0.100783,Overprediction
JNJ,-0.039784,0.056192,0.095976,Overprediction
UBER,0.083001,0.174668,0.091667,Overprediction
QCOM,-0.096174,-0.007512,0.088663,Overprediction


In [ ]:
# Guardar dataframe con cluster con fecha en el nombre del archivo

dia = datetime.now().day
mes = datetime.now().month
year = datetime.now().year

# Crear carpeta resultados si no existe
if not os.path.exists(f"{data_folder}/resultados"):
    os.makedirs(f"{data_folder}/resultados")

resultados_agrupados.to_csv(f"{data_folder}/resultados/cluster_data_{year}_{mes}_{dia}.csv", index=False)

## Anexo: optimizacion de hiperparametros

* La siguiente celda se utilizó para optimizar los hiperparámetros de los modelos de forma individual. La ejecución de la misma no es necesaria.
* En caso de utilizarla, reemplazar 'no' por 'si' en la primer línea. Puede demorar varios minutos.

In [12]:
ejecutar_celda = 'no' # Reemplazar para ejecutar
modelo_a_optimizar = 1  # 1 = Random Forest 

if ejecutar_celda == 'si':

    if modelo_a_optimizar == 1:
        nombre_modelo= 'Random Forest'
        print(f"Configurando GridSearchCV para {nombre_modelo}")

        # Pipeline usando el preprocesador específico para Random Forest
        modelo_base = Pipeline(steps=[
            ('preprocesador', preprocessor),
            ('rf', RandomForestRegressor(random_state=42))
        ])

        param_grid = {
            'rf__n_estimators': [300],
            'rf__max_depth': [10],
            'rf__min_samples_leaf': [3],
            'rf__min_samples_split': [5],
            'rf__max_features': ['sqrt', 'log2', 0.5],
            'rf__max_samples': [0.7]
        }

    elif modelo_a_optimizar == 2: # Por implementar
        nombre_modelo= 'LightGBM'
        print(f"Configurando GridSearchCV para {nombre_modelo}")

        # Pipeline usando el preprocesador específico para LightGBM
        modelo_base = Pipeline(steps=[
            ('preprocesador', preprocesado_rf),
            ('lgb', lgb.LGBMRegressor(random_state=42,
                                      subsample=0.8,
                                      colsample_bytree=0.8
))
        ])

        param_grid = {
            'lgb__n_estimators': [500, 550, 600],
            'lgb__max_depth': [5],
            'lgb__learning_rate': [0.05],
            'lgb__num_leaves': [20],
            'lgb__min_child_samples': [20]
        }

    else:
        raise ValueError("Opción no válida. Por favor, selecciona 1 o 2.")

    # Configurar el GridSearchCV
    grid_search = GridSearchCV(
        estimator=modelo_base,
        param_grid=param_grid,
        scoring='neg_mean_absolute_error',
        cv=3,
        n_jobs=-1,
        verbose=2
    )

    # Entrenar con X_train y y_train originales
    print(f"Iniciando búsqueda de hiperparámetros. Esto tomará unos minutos.")
    grid_search.fit(X_train, y_train)

    # Resultados
    print("\n--- Búsqueda Finalizada ---")
    print("Mejores hiperparámetros encontrados:")
    print(grid_search.best_params_)

    # Extraer el mejor modelo
    best_model = grid_search.best_estimator_

    # Predecir en el set de prueba usando X_test original
    y_pred = best_model.predict(X_test)

    print("\nEvaluación en el set de Prueba:\n", obtener_metricas(y_test, y_pred, nombre_modelo))